## Chroma Vector Database with LangChain

This notebook tests creating a **Chroma** vector store from a PDF document. Steps:
1. Load the PDF from the `00_data` folder using `PyPDFLoader`.
2. Split it into chunks with a text splitter.
3. Embed the chunks using HuggingFace sentence-transformer embeddings.
4. Store the embeddings in a local Chroma DB and run a similarity search query against it.

In [1]:
import os
from dotenv import load_dotenv

# Change working directory to the parent folder (01_Langchain)
os.chdir(os.path.abspath(".."))
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("00_data/attention.pdf")
docs = loader.load()
print(f"Number of pages loaded: {len(docs)}")
docs[0]

/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_18665/3750619610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/bk/brew-global-venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of pages loaded: 15


Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '00_data/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nl

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_docs = text_splitter.split_documents(docs)
print(f"Number of chunks: {len(split_docs)}")

Number of chunks: 52


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8249.09it/s]


In [5]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db",
)
print("Chroma vector store created with", vectordb._collection.count(), "documents")

Chroma vector store created with 52 documents


In [6]:
query = "What is self-attention?"
results = vectordb.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"--- Result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

--- Result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Result 2 (page 5) ---
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encounter

--- Result 3 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
t



## Using the Vector Store as a Retriever

`similarity_search` above is a Chroma-specific method. A **retriever** wraps the vector store in LangChain's standard `Runnable` interface, so the same object can be dropped into a chain or a RAG pipeline regardless of which vector store is behind it.

The search behaviour is configured once, when the retriever is created:

- `search_type="similarity"` (default) returns the `k` nearest chunks.
- `search_type="mmr"` uses Maximal Marginal Relevance to trade some similarity for diversity, which helps when the top hits are near-duplicates of each other.
- `search_type="similarity_score_threshold"` drops anything below a relevance score, so a weak query can return fewer than `k` chunks - or none at all.

### 1. `similarity` (the default)

Returns the `k` chunks whose embeddings sit closest to the query embedding - the same ranking as the `similarity_search` call above, just reached through the retriever interface.

In [7]:
# Wrap the Chroma store in the standard retriever interface
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Document {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

Retrieved 3 documents
--- Document 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- Document 2 (page 5) ---
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encounter

--- Document 3 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
t



### 2. `mmr` (Maximal Marginal Relevance)

MMR fetches a larger candidate pool (`fetch_k`), then greedily picks `k` chunks that are relevant to the query *and* unlike the chunks already picked. It is worth using when the top hits are near-duplicates of each other, which wastes context window in a RAG prompt. Compare the pages it returns against the plain similarity results above.

In [8]:
# MMR re-ranks a larger candidate pool (fetch_k) down to k diverse results
# lambda_mult: 1.0 = maximum relevance, 0.0 = maximum diversity
mmr_retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 20, "lambda_mult": 0.5},
)

for i, doc in enumerate(mmr_retriever.invoke(query)):
    print(f"--- MMR result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:300])
    print()

--- MMR result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

--- MMR result 2 (page 12) ---
Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
h

--- MMR result 3 (page 5) ---
executed operations, whereas a recurrent layer requires O(n) sequential operations. In terms of
computational complexity, self-attention layers are faster than recurrent layers when the sequence
6



### 3. `similarity_score_threshold`

Drops any chunk whose **relevance score** falls below `score_threshold`, so a weak query can return fewer than `k` chunks - or none at all. This is what stops an off-topic question from stuffing irrelevant context into a RAG prompt.

Relevance scores are not on a universal scale. LangChain converts the store's raw distance into a 0-1 score, and the conversion depends on the distance metric and on whether the embeddings are unit-normalised. Chroma defaults to L2 distance here, and with `all-MiniLM-L6-v2` the scores land roughly between -0.3 and 0.5 - so a threshold of 0.8 would silently return nothing at all. Always look at the real scores before choosing a threshold.

In [9]:
# Inspect the real score range before picking a threshold
off_topic_query = "How do I bake sourdough bread?"

for q in (query, off_topic_query):
    scored = vectordb.similarity_search_with_relevance_scores(q, k=5)
    print(f"{q!r}")
    print("   scores:", [round(float(score), 3) for _, score in scored])

'What is self-attention?'
   scores: [0.303, 0.244, 0.24, 0.228, 0.218]
'How do I bake sourdough bread?'
   scores: [-0.238, -0.25, -0.268, -0.3, -0.306]


/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_18665/1213740318.py:5: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='7b7422c0-30f9-4508-adc0-01409f12da66', metadata={'trapped': '/False', 'page_label': '9', 'page': 8, 'title': '', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'author': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'total_pages': 15, 'source': '00_data/attention.pdf', 'creationdate': '2023-08-03T00:07:29+00:00', 'moddate': '2023-08-03T00:07:29+00:00'}, page_content='Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metrics are on the English-to-German translation development set, newstest2013. Listed\nperplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to\nper-word perplexities.\nN d model dff h d k dv Pdrop ϵl

In [10]:
threshold_retriever = vectordb.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.25},
)

# The off-topic query returns nothing, and LangChain logs a warning saying so
for q in (query, off_topic_query):
    docs = threshold_retriever.invoke(q)
    print(f"{q!r} -> {len(docs)} document(s) above the threshold")
    for doc in docs:
        print(f"   page {doc.metadata.get('page')}: {doc.page_content[:120]}...")
    print()

/Users/bk/brew-global-venv/lib/python3.14/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='7b7422c0-30f9-4508-adc0-01409f12da66', metadata={'total_pages': 15, 'page': 8, 'author': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'source': '00_data/attention.pdf', 'page_label': '9', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': '', 'subject': '', 'trapped': '/False', 'moddate': '2023-08-03T00:07:29+00:00', 'creationdate': '2023-08-03T00:07:29+00:00', 'producer': 'pdfTeX-1.40.25'}, page_content='Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metrics are on the English-to-German translation development set, newstest2013. Listed\nperplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to\nper-word perplexities.\nN d model dff h

'What is self-attention?' -> 1 document(s) above the threshold
   page 2: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where...

'How do I bake sourdough bread?' -> 0 document(s) above the threshold



### 4. Metadata filtering

A filter is not a search type - it is a constraint layered on top of any of them. Chroma evaluates it **inside the collection**, so the vector search only ever considers matching chunks; `k` still means "k results", not "k candidates, then filtered down".

Chroma exposes two filters:

- `filter` (Chroma's `where`) matches on metadata. `PyPDFLoader` gives every chunk `page`, `source`, `total_pages` and the PDF's own fields, so all of those are filterable. Operators: `$eq $ne $gt $gte $lt $lte $in $nin`, combined with `$and` / `$or`.
- `where_document` matches on the chunk **text**. `{"$contains": "Multi-Head"}` keeps only chunks containing that exact substring. This is a filter, not ranked keyword search - the substring is present or absent, with no scoring. For real lexical ranking see BM25 below.

In [11]:
# Metadata filter: restrict the search to page 2
for doc in vectordb.similarity_search(query, k=3, filter={"page": 2}):
    print(f"page {doc.metadata['page']}: {doc.page_content[:90]}...")
print()

# Operators work too
print("pages >= 5      :", [d.metadata["page"] for d in vectordb.similarity_search(query, k=3, filter={"page": {"$gte": 5}})])

# where_document filters on the chunk text (exact substring, not ranked)
print("has 'Multi-Head':", [d.metadata["page"] for d in vectordb.similarity_search(query, k=3, where_document={"$contains": "Multi-Head"})])

# Filters pass straight through the retriever interface as well
filtered_retriever = vectordb.as_retriever(search_kwargs={"k": 3, "filter": {"page": {"$lte": 3}}})
print("via retriever   :", [d.metadata["page"] for d in filtered_retriever.invoke(query)])

page 2: 3.2 Attention
An attention function can be described as mapping a query and a set of key-v...
page 2: itself. To facilitate these residual connections, all sub-layers in the model, as well as ...
page 2: Figure 1: The Transformer - model architecture.
The Transformer follows this overall archi...

pages >= 5      : [5, 12, 5]
has 'Multi-Head': [1, 3, 3]
via retriever   : [2, 1, 2]


### 5. Distance metric (an index setting, not a search type)

Cosine similarity often gets listed alongside "similarity" and "MMR" as though it were a third search mode. It is not - it is the **distance function the index uses**, chosen once when the collection is created and applied by every search afterwards. Chroma's `hnsw:space` accepts `"l2"` (the default), `"cosine"` and `"ip"` (inner product).

For `all-MiniLM-L6-v2` the embeddings come out unit-normalised, and for unit vectors L2 distance and cosine are monotonically related - so **switching metric does not change the ranking here**. What it changes is the *score scale*, and that matters for `similarity_score_threshold`: under `l2` the relevance scores ran from about -0.3 to 0.5, while under `cosine` they sit inside 0-1 the way the documentation assumes.

In [12]:
# Both stores are built fresh here so the comparison is apples-to-apples
# (vectordb persists to ./chroma_db and gains a duplicate copy of every chunk
#  each time this notebook is re-run from the top)
l2_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="attention_l2",
    collection_metadata={"hnsw:space": "l2"},  # the default
)
cosine_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name="attention_cosine",
    collection_metadata={"hnsw:space": "cosine"},
)

for label, store in (("l2 (default)", l2_db), ("cosine", cosine_db)):
    on = store.similarity_search_with_relevance_scores(query, k=3)
    off = store.similarity_search_with_relevance_scores(off_topic_query, k=3)
    print(f"{label:14} on-topic  {[round(float(s), 3) for _, s in on]}")
    print(f"{'':14} off-topic {[round(float(s), 3) for _, s in off]}")

print()
print("l2 pages    :", [d.metadata["page"] for d in l2_db.similarity_search(query, k=5)])
print("cosine pages:", [d.metadata["page"] for d in cosine_db.similarity_search(query, k=5)])
print("-> identical ranking, different score scale")

l2 (default)   on-topic  [0.303, 0.244, 0.24]
               off-topic [-0.238, -0.25, -0.268]
cosine         on-topic  [0.507, 0.466, 0.462]
               off-topic [0.125, 0.116, 0.104]

l2 pages    : [2, 5, 1, 12, 5]
cosine pages: [2, 5, 1, 12, 5]
-> identical ranking, different score scale


/var/folders/j5/52bx342j7k59wyrt9n978pp00000gn/T/ipykernel_18665/338616129.py:19: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='a7e9d9d9-9d2a-43a7-8d70-ff0362cbc706', metadata={'title': '', 'moddate': '2023-08-03T00:07:29+00:00', 'creationdate': '2023-08-03T00:07:29+00:00', 'source': '00_data/attention.pdf', 'keywords': '', 'trapped': '/False', 'page': 8, 'page_label': '9', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'total_pages': 15, 'subject': '', 'author': ''}, page_content='Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base\nmodel. All metrics are on the English-to-German translation development set, newstest2013. Listed\nperplexities are per-wordpiece, according to our byte-pair encoding, and should not be compared to\nper-word perplexities.\nN d model dff h d k dv Pdrop ϵl

### 6. Keyword search (BM25)

Everything above is **dense** retrieval: the query and the chunks become vectors, and closeness in vector space stands in for meaning. That fails on the things embeddings smooth away - exact product codes, error numbers, rare proper nouns, a specific acronym. If you ask for `BLEU` and the model has never seen it, no amount of `k` will help.

**BM25** is the classic **sparse**, lexical alternative: it scores on term overlap, weighting rare terms higher and long documents lower. No embedding model and no vector store involved - it runs over the chunk list directly.

Requires `pip install rank_bm25`.

In [13]:
from langchain_community.retrievers import BM25Retriever

# BM25 indexes the raw text - no embeddings, no vector store
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = 3

for i, doc in enumerate(bm25_retriever.invoke(query)):
    print(f"--- BM25 result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

--- BM25 result 1 (page 5) ---
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations required.
The third is th

--- BM25 result 2 (page 2) ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

--- BM25 result 3 (page 14) ---
Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
sh



### 7. Hybrid search (dense + sparse)

Hybrid search runs a dense retriever and a sparse one over the same corpus and merges the two ranked lists. It catches both the paraphrase the keyword search misses and the exact term the embedding blurs.

`EnsembleRetriever` merges with **Reciprocal Rank Fusion**: each document scores `sum(weight / (60 + rank))` across the lists it appears in, so a chunk that both retrievers rank highly wins, and the two score scales - which are not comparable - never have to be reconciled.

Note it returns the *union* of the result sets, so asking two retrievers for 3 documents each can yield up to 6. Set `k` on the individual retrievers to control the size.

On LangChain 1.x this lives in `langchain_classic.retrievers`, **not** `langchain.retrievers` - most tutorials online still show the old path, which no longer exists.

In [14]:
from langchain_classic.retrievers import EnsembleRetriever

dense_retriever = vectordb.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],  # must sum to 1.0; tilt towards whichever suits your corpus
)

print("keyword only:", [d.metadata.get("page") for d in bm25_retriever.invoke(query)])
print("dense only  :", [d.metadata.get("page") for d in dense_retriever.invoke(query)])
print("hybrid      :", [d.metadata.get("page") for d in hybrid_retriever.invoke(query)])
print()

for i, doc in enumerate(hybrid_retriever.invoke(query)):
    print(f"--- Hybrid result {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

keyword only: [5, 2, 14]
dense only  : [2, 5, 1]
hybrid      : [2, 5, 1, 5, 2, 14]

--- Hybrid result 1 (page 2) ---
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as 

--- Hybrid result 2 (page 5) ---
PE pos.
We also experimented with using learned positional embeddings [9] instead, and found that the two
versions produced nearly identical results (see Table 3 row (E)). We chose the sinusoidal vers

--- Hybrid result 3 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is


--- Hybrid result 4 (page 5) ---
One is the total computational complexity per layer. Another is the amount of computation that can
be parallelized, as measured by the minimum number of sequential operations req

### 8. Reranking (cross-encoder)

Retrieval embeds the query and the chunk *separately* - the model never sees them together, which is what makes it fast enough to index thousands of chunks in advance. A **cross-encoder** does the opposite: it reads the query and one chunk as a single input and scores the pair directly. Far more accurate, far too slow to run over the whole corpus.

So they get combined: retrieve a wide net with the bi-encoder (`k=20`), then rerank and keep the best few. This is usually the largest quality gain per line of code in a RAG pipeline.

`ContextualCompressionRetriever` is the generic wrapper - a base retriever plus a compressor. `CrossEncoderReranker` is one compressor; others trim or summarise the chunks instead.

In [15]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Downloads a small reranking model (~90 MB) on first run
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

# Cast a wide net, then let the cross-encoder pick the best 3
wide_retriever = vectordb.as_retriever(search_kwargs={"k": 20})
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=CrossEncoderReranker(model=cross_encoder, top_n=3),
    base_retriever=wide_retriever,
)

print("before rerank:", [d.metadata.get("page") for d in wide_retriever.invoke(query)][:6], "...")
print("after rerank :", [d.metadata.get("page") for d in rerank_retriever.invoke(query)])
print()

for i, doc in enumerate(rerank_retriever.invoke(query)):
    print(f"--- Reranked {i+1} (page {doc.metadata.get('page')}) ---")
    print(doc.page_content[:200])
    print()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 10418.78it/s]


before rerank: [2, 5, 1, 12, 5, 4] ...
after rerank : [1, 4, 4]

--- Reranked 1 (page 1) ---
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difficult to learn dependencies between distant positions [ 12]. In the Transformer this is


--- Reranked 2 (page 4) ---
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward
i

--- Reranked 3 (page 4) ---
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from t



### 9. Score and by-vector variants of the search API

`similarity_search` returns bare `Document`s. Chroma exposes three more variants of the same search, and the two kinds of "score" sort in **opposite directions**:

| Method | Returns | Score meaning |
|---|---|---|
| `similarity_search` | `[Document]` | - |
| `similarity_search_with_score` | `[(Document, float)]` | **Raw Chroma distance. Lower is closer.** |
| `similarity_search_with_relevance_scores` | `[(Document, float)]` | Converted to 0-1. **Higher is closer.** |
| `similarity_search_by_vector` | `[Document]` | - |
| `similarity_search_by_vector_with_relevance_scores` | `[(Document, float)]` | **Raw distance, despite the name.** |

Two traps here:

1. The raw distance depends on `hnsw:space` (section 5). In the default `l2` space it is a *squared* L2 distance, so the numbers are not comparable to FAISS's, and only `similarity_search_with_relevance_scores` gives you the normalised 0-1 value that `similarity_score_threshold` filters on.
2. `similarity_search_by_vector_with_relevance_scores` does **not** return relevance scores in spite of its name - it returns the same raw distances as `similarity_search_with_score`. Run the cell and compare the numbers.

The `_by_vector` forms skip the embedding step and take a vector you already have: handy for reusing one query embedding, for a vector from elsewhere, or for "more like this" - feeding an existing chunk's own embedding back in as the query.

In [16]:
# Raw distance: LOWER is closer
print("similarity_search_with_score  (raw distance, lower = closer)")
for doc, score in vectordb.similarity_search_with_score(query, k=3):
    print(f"   distance {score:.4f}   page {doc.metadata.get('page')}   {doc.page_content[:60]}...")

# Converted relevance: HIGHER is closer. Same documents, opposite ordering of the number.
print()
print("similarity_search_with_relevance_scores  (0-1 scale, higher = closer)")
for doc, score in vectordb.similarity_search_with_relevance_scores(query, k=3):
    print(f"   relevance {score:.4f}  page {doc.metadata.get('page')}")

# Scores honour filters like any other search
print()
print("with_score, pages >= 5:",
      [(d.metadata["page"], round(float(s), 4))
       for d, s in vectordb.similarity_search_with_score(query, k=3, filter={"page": {"$gte": 5}})])

similarity_search_with_score  (raw distance, lower = closer)
   distance 0.9862   page 2   3.2 Attention
An attention function can be described as mapp...
   distance 1.0689   page 5   PE pos.
We also experimented with using learned positional e...
   distance 1.0755   page 1   in the distance between positions, linearly for ConvS2S and ...

similarity_search_with_relevance_scores  (0-1 scale, higher = closer)
   relevance 0.3027  page 2
   relevance 0.2442  page 5
   relevance 0.2395  page 1

with_score, pages >= 5: [(5, 1.0689), (12, 1.0914), (5, 1.1063)]


In [17]:
# Embed the query once, then search with the vector directly
query_vector = embeddings.embed_query(query)

print("by vector:", [d.metadata.get("page") for d in vectordb.similarity_search_by_vector(query_vector, k=3)])
print("by text  :", [d.metadata.get("page") for d in vectordb.similarity_search(query, k=3)])
print("-> identical; similarity_search just calls embed_query for you")

# Despite "relevance_scores" in the name, these are the raw distances again
print()
print("similarity_search_by_vector_with_relevance_scores:")
for doc, score in vectordb.similarity_search_by_vector_with_relevance_scores(query_vector, k=3):
    print(f"   {score:.4f}  page {doc.metadata.get('page')}   <- same number as with_score above")

# "More like this": use an existing chunk's own embedding as the query.
# The chunk itself comes back first, at distance ~0.
seed_doc = split_docs[10]
seed_vector = embeddings.embed_query(seed_doc.page_content)

print()
print(f"seed chunk (page {seed_doc.metadata.get('page')}): {seed_doc.page_content[:70]}...")
for doc, score in vectordb.similarity_search_by_vector_with_relevance_scores(seed_vector, k=3):
    print(f"   distance {score:.4f}   page {doc.metadata.get('page')}   {doc.page_content[:60]}...")

by vector: [2, 5, 1]
by text  : [2, 5, 1]
-> identical; similarity_search just calls embed_query for you

similarity_search_by_vector_with_relevance_scores:
   0.9862  page 2   <- same number as with_score above
   1.0689  page 5   <- same number as with_score above
   1.0755  page 1   <- same number as with_score above

seed chunk (page 2): Figure 1: The Transformer - model architecture.
The Transformer follow...
   distance 0.0000   page 2   Figure 1: The Transformer - model architecture.
The Transfor...
   distance 0.4174   page 2   itself. To facilitate these residual connections, all sub-la...
   distance 0.4880   page 4   The Transformer uses multi-head attention in three different...


### 10. Reading from the persisted database

Because `vectordb` was created with a `persist_directory`, Chroma has been writing to `./chroma_db` all along - there is no `save` step, unlike FAISS's `save_local`. To reopen it later you construct `Chroma` directly instead of calling `from_documents`:

```python
Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
```

No `documents` argument, and nothing is re-embedded - it just attaches to the collection already on disk. The embedding function is still required, because it embeds the *queries* you run against it. Pass the same model used to build the store: nothing validates this, and a mismatch gives silently meaningless results.

> **This is also the fix for a bug in this notebook.** `Chroma.from_documents` **appends**, so every top-to-bottom run adds another full copy of all 52 chunks and duplicates start showing up in the results. Check the count the cell below prints. To make the notebook re-runnable, have cell 5 open the existing store when `./chroma_db` is already there and only call `from_documents` when it is not.

Beyond searching, `get()` reads rows straight out of the collection with no query and no embedding at all - the equivalent of FAISS's docstore lookup, but with filtering built in.

In [18]:
import os

CHROMA_PATH = "./chroma_db"
print("on disk:", sorted(os.listdir(CHROMA_PATH))[:6])

# Open the existing collection - no documents, no re-embedding
persisted_db = Chroma(
    persist_directory=CHROMA_PATH,
    embedding_function=embeddings,
)

count = persisted_db._collection.count()
print("documents in the persisted collection:", count)
print(f"-> {count // 52} copy/copies of the 52 chunks"
      f"{' - re-running this notebook appended duplicates' if count > 52 else ''}")

print()
print("search on the reopened store:", [d.metadata.get("page") for d in persisted_db.similarity_search(query, k=3)])

on disk: ['ac4b4611-d02f-47aa-a172-5b71930d99b6', 'chroma.sqlite3']
documents in the persisted collection: 52
-> 1 copy/copies of the 52 chunks

search on the reopened store: [2, 5, 1]


In [19]:
# .get() reads rows directly out of the collection - no query, no embedding
records = persisted_db.get(limit=3)
print("returned keys:", sorted(records.keys()))
print()

for doc_id, text, meta in zip(records["ids"], records["documents"], records["metadatas"]):
    print(f"--- id {doc_id[:8]}... | page {meta.get('page')} ---")
    print(text[:180])
    print()

# get() takes the same metadata filters as a search
print("chunks on page 3:", len(persisted_db.get(where={"page": 3})["ids"]))

# Stored vectors are not returned unless you ask for them
with_vectors = persisted_db.get(limit=1, include=["embeddings"])
print("stored vector dim:", len(with_vectors["embeddings"][0]))

# get_by_ids returns Document objects rather than raw dicts
print("as Documents:", [d.metadata.get("page") for d in persisted_db.get_by_ids(records["ids"])])

returned keys: ['data', 'documents', 'embeddings', 'ids', 'included', 'metadatas', 'uris']

--- id ac86d4c4... | page 0 ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attent

--- id 4890e80d... | page 0 ---
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on

--- id 515d056e... | page 0 ---
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limite

chunks on page 3: 3
stored vector dim: 384
as Documents: [0, 0, 0]


### Where each technique fits

| Technique | Layer | Use it when |
|---|---|---|
| `similarity` | search type | Default; you want the `k` closest chunks. |
| `mmr` | search type | Top hits are near-duplicates and waste prompt space. |
| `similarity_score_threshold` | search type | Off-topic questions must return nothing rather than noise. |
| Metadata filter | query option | You can narrow by page, source, date, tenant, permissions. |
| Distance metric | index setting | Fixed when the index is built; affects the score scale. |
| BM25 | separate retriever | Exact terms, codes, rare names the embedding blurs. |
| Hybrid / RRF | composition | Production RAG defaults - you want both of the above. |
| Cross-encoder rerank | post-processing | Precision matters and you can afford one extra model pass. |

Still further up the stack, and all in `langchain_classic.retrievers`: `MultiQueryRetriever` (an LLM rewrites the query several ways and unions the hits), `ParentDocumentRetriever` (embed small chunks, return their larger parent), `SelfQueryRetriever` (an LLM turns "papers after 2020 about attention" into a metadata filter), and `MultiVectorRetriever` (index summaries or hypothetical questions, return the source chunk). Each needs an LLM or extra indexing, so they are out of scope here.